In [52]:
#importing required libraries
!pip3 install  langchain langchain_community chromadb  langchain-google-vertexai
!pip3 install "unstructured[all-docs]" pypdf pillow pydantic lxml pillow matplotlib chromadb tiktoken
#checking dependencies

In [53]:
#initializing..
import os
import sys

from dotenv import load_dotenv
load_dotenv()

PROJECT_ID=os.getenv("PROJECT_ID")
REGION=os.getenv("REGION")

from google.colab import auth
auth.authenticate_user()

import vertexai
vertexai.init(project=PROJECT_ID,location=REGION)

In [24]:
# #basic example
# import logging
# import zipfile
# import requests

# # importing required configs
# logging.basicConfig(level=logging.INFO)
# DATA_URL=os.getenv("DATA_URL")

# #requesting..
# response=requests.get(DATA_URL)
# filename='resources/resources.zip'
# with open(filename,"wb") as file:
#     file.write(response.content)



# #extracting current zip filter
# with zipfile.ZipFile(filename,"r") as zip_ref:
#     zip_ref.extractall()

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# #defining requiured libraries
# import os
# import sys


# paths=['/content/Dataset/zararli_organizmalar_ve_latince_katalogu.pdf','/content/Dataset/bitki_koruma_ve_mucadele_yontemleri.pdf']
# os.makedirs("Dataset")

# for path in paths:
#   import shutil
#   if os.path.exists(path):
#     shutil.move(path,'Dataset')



# New Section

In [26]:
#all the page
import zipfile
import os

zip_path = "/content/Dataset/resource1.zip"
extract_path = "Dataset/extracted"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

pdf_files = []

for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, file))

print(pdf_files)

from langchain_community.document_loaders import PyPDFLoader

docs = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs.extend(loader.load())

print(f"Total pages loaded: {len(docs)}")



['Dataset/extracted/bitki_koruma_ve_mucadele_yontemleri.pdf', 'Dataset/extracted/zararli_organizmalar_ve_latince_katalogu.pdf']
Total pages loaded: 5


In [27]:
tables=[]
texts = [doc.page_content for doc in docs]

print(texts[:2])

["BİTKİ KORUMA VE MÜCADELE YÖNTEMLERİ REHBERİ\nRAG BİLGİ TABANI\nBitki Sağlığı Prensipleri ve Uygulama Esasları\nSayfa 1\nBİTKİ KORUMA VE MÜCADELE YÖNTEMLERİ\nREHBERİ\nEntegre Zararlı Yönetimi, Sürvey, Karantina ve Kalıntı Denetimi Prensipleri\n1. ENTEGRE ZARARLI YÖNETİMİ (IPM) VE TEMEL İLKELER\nEntegre Zararlı Yönetimi (IPM); kültür bitkilerinde zararlı türlerin popülasyon dinamikleri ve çevre ile ilişkilerini\ndikkate alarak, uygun bütün mücadele metotlarını ve tekniklerini uyumlu bir şekilde kullanan, zararlı\npopülasyonunu Ekonomik Zarar Seviyesi'nin altında tutan bir agroekosistem yönetim sistemidir.\nIPM uygulamalarında zararlıların Ekonomik Zarar Eşiği (EZE), insan sağlığı, çevre ve doğal denge ön planda\ntutulur. Kimyasal mücadele her zaman en son çare olarak değerlendirilir. Bu sistem Entegre Ürün Yönetimi, İyi\nTarım Uygulamaları (İTU) ve sürdürülebilir tarımsal üretimin temelini oluşturmaktadır.\nIPM Temel Amaçları: Periyodik arazi kontrolleri alışkanlığı kazandırmak, doğal 

In [28]:
#checking contents
print("First:/t",texts[0])
print("Length:/t",len(texts))

First:/t BİTKİ KORUMA VE MÜCADELE YÖNTEMLERİ REHBERİ
RAG BİLGİ TABANI
Bitki Sağlığı Prensipleri ve Uygulama Esasları
Sayfa 1
BİTKİ KORUMA VE MÜCADELE YÖNTEMLERİ
REHBERİ
Entegre Zararlı Yönetimi, Sürvey, Karantina ve Kalıntı Denetimi Prensipleri
1. ENTEGRE ZARARLI YÖNETİMİ (IPM) VE TEMEL İLKELER
Entegre Zararlı Yönetimi (IPM); kültür bitkilerinde zararlı türlerin popülasyon dinamikleri ve çevre ile ilişkilerini
dikkate alarak, uygun bütün mücadele metotlarını ve tekniklerini uyumlu bir şekilde kullanan, zararlı
popülasyonunu Ekonomik Zarar Seviyesi'nin altında tutan bir agroekosistem yönetim sistemidir.
IPM uygulamalarında zararlıların Ekonomik Zarar Eşiği (EZE), insan sağlığı, çevre ve doğal denge ön planda
tutulur. Kimyasal mücadele her zaman en son çare olarak değerlendirilir. Bu sistem Entegre Ürün Yönetimi, İyi
Tarım Uygulamaları (İTU) ve sürdürülebilir tarımsal üretimin temelini oluşturmaktadır.
IPM Temel Amaçları: Periyodik arazi kontrolleri alışkanlığı kazandırmak, doğal düşmanl

In [29]:
#testing current function
promptText=""" You are a scientific research assistant specializing in invasive species, plant protection, ecology, and plant pathology.

Your task is to analyze the provided document and generate a detailed, accurate, and comprehensive summary suitable for a Retrieval-Augmented Generation (RAG) system.

## Instructions

1. Read and understand the entire input document.
2. Identify the main subject, objectives, and important findings.
3. Preserve all scientifically relevant information.
4. Do not introduce external information or unsupported claims.
5. Maintain scientific terminology and Latin species names.
6. Preserve geographical locations, dates, numerical values, and experimental findings.
7. Distinguish confirmed scientific findings from hypotheses and assumptions.
8. Do not omit important details simply to make the summary shorter.
9. If information is missing, explicitly state that it is not specified.
10. Maintain references to the original document whenever available.

## Summary Structure

Organize the summary using the following sections:

### 1. General Overview
Explain the document's main topic, purpose, research objectives, and scope.

### 2. Invasive Species
Identify the invasive species discussed, including:
- Scientific and common names.
- Biological characteristics.
- Native and introduced distribution.
- Reproduction and spreading mechanisms.
- Ecological characteristics.

### 3. Host Plants
Identify affected plant species, including:
- Scientific and common names.
- Susceptible plant species.
- Plant parts affected.
- Host-specificity information.

### 4. Geographic Distribution
Summarize:
- Countries and regions where the species has been reported.
- Confirmed geographical presence.
- Potential distribution.
- Environmental conditions affecting distribution.

Clearly distinguish confirmed presence from predicted or potential distribution.

### 5. Symptoms and Damage
Describe:
- Visible infestation symptoms.
- Physical damage to plants.
- Ecological consequences.
- Economic consequences.
- Early detection indicators.

### 6. Prevention and Control Methods
Summarize scientifically supported recommendations:
- Preventive measures.
- Monitoring and early detection.
- Mechanical and cultural control.
- Biological control.
- Chemical control.
- Long-term management strategies.

Include application conditions and limitations whenever available.

### 7. Scientific Findings
Preserve important experimental results, statistics, observations, and research conclusions.

### 8. Limitations and Uncertainties
Identify research limitations, missing information, contradictory findings, and unresolved questions.

### 9. Keywords
Generate relevant keywords for semantic search, including scientific names, host plants, geographical locations, symptoms, and prevention methods.

### 10. Final Summary
Provide a coherent synthesis of the document's most important findings and their relevance to plant protection.

## Important Requirements

- Prioritize scientific accuracy over brevity.
- Do not hallucinate information.
- Do not confuse invasive species with native species.
- Do not confuse confirmed distribution with potential distribution.
- Preserve the relationship between invasive species, host plants, geographical locations, and control methods.
- Do not generate recommendations that are not supported by the document.
- Write the summary in English.
- Use clear headings and structured paragraphs.

INPUT DOCUMENT:

{document_content}"""

In [41]:
# Generating text summaries

# Required libraries
!pip install -q langchain langchain_community chromadb langchain-google-vertexai
!pip install -q -U langchain-google-genai
!gcloud services enable aiplatform.googleapis.com --project=virtual-sylph-514013-h5

import warnings
warnings.filterwarnings("ignore")



from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
LOCATION=os.getenv("LOCATION")
PROJECT_ID=os.getenv("PROJECT_ID")

def summaryFromInput(texts, tables, promptText, summarizeTexts=False,
                      modelName='gemini-3-flash-preview'):

    prompt = PromptTemplate.from_template(promptText)

    empty = RunnableLambda(
        lambda x: AIMessage(content="Error processing document")
    )


    model = ChatGoogleGenerativeAI(
        model=modelName,
        google_api_key=GOOGLE_API_KEY,
        temperature=0,
        max_output_tokens=8192
    ).with_fallbacks([empty])

    summary = {"document_content": lambda x: x} | prompt | model | StrOutputParser()

    # Summaries
    textSummaries = []
    tableSummaries = []

    if texts:
        textSummaries = summary.batch(texts, config={"max_concurrency": 1})

    if tables:
        tableSummaries = summary.batch(tables, config={"max_concurrency": 1})

    return textSummaries, tableSummaries


# Usage
textSummaries, tableSummaries = summaryFromInput(
    texts=texts,
    tables=tables,
    promptText=promptText
)

In [42]:
import warnings
warnings.filterwarnings("ignore")
textSummaries,tableSummaries=summaryFromInput(texts=texts,tables=tables,promptText=promptText)

In [46]:
try:
    import uuid
    from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
    from langchain_classic.storage import InMemoryStore
    from langchain_chroma import Chroma
    from langchain_core.documents import Document
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.prompts import PromptTemplate
    from langchain_core.messages import AIMessage
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.runnables import RunnableLambda
    print("✅ All imports succeeded")
except ImportError as e:
    print(f"❌ Missing: {e}")

✅ All imports succeeded


In [51]:
!pip3 install -q -U langchain-huggingface sentence-transformers
try:
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError as e:
    print("Missing:\t", e)

def create_multi_vector_retriever(
    vectorstore, text_summaries, texts, table_summaries, tables, image_summaries, images
):
    """
    Create retriever that indexes summaries, but returns raw images or texts
    """
    store = InMemoryStore()
    id_key = "doc_id"

    # Actually create the retriever — this was missing
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=store,
        id_key=id_key,
    )

    def add_documents(retriever, doc_summaries, doc_contents):
        doc_ids = [str(uuid.uuid4()) for _ in doc_contents]
        summary_docs = [
            Document(page_content=s, metadata={id_key: doc_ids[i]})
            for i, s in enumerate(doc_summaries)
        ]
        retriever.vectorstore.add_documents(summary_docs)
        retriever.docstore.mset(list(zip(doc_ids, doc_contents)))

    if text_summaries:
        add_documents(retriever, text_summaries, texts)
    if table_summaries:
        add_documents(retriever, table_summaries, tables)
    if image_summaries:
        add_documents(retriever, image_summaries, images)

    return retriever


# The vectorstore to use to index the summaries
vectorstore = Chroma(
    collection_name="mm_rag_cj_blog",
    embedding_function=HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        encode_kwargs={"normalize_embeddings": True},
    )
)

# Create retriever
retriever = create_multi_vector_retriever(
    vectorstore,
    text_summaries,
    texts,
    table_summaries,
    tables,
    image_summaries,
    img_base64_list,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

NameError: name 'text_summaries' is not defined

In [ ]:
question=''
output=retriever.invoke(question)